In [2]:
# test all sample histories

In [33]:
from nse import nse_json, clean_eq_hist, clean_index_hist, clean_opt_hist
import datetime
import pandas as pd
import datetime

In [43]:
import yaml
with open('refmap.yml', 'rb') as ref:
    def_dict = yaml.safe_load(ref)

# Testing Single Static URLs 

In [45]:
dt_raw = def_dict['NSE']['EXPIRY_DAY_OLD']['OLD_DATE']


In [56]:
df = pd.read_csv('https://api.kite.trade/instruments')
# df[(df.exchange == 'NFO') & (df.name == 'NIFTY')]
df


,instrument_token,exchange_token,tradingsymbol,name,last_price,expiry,strike,tick_size,lot_size,instrument_type,segment,exchange
0,542521606,2119225,EURINR23AUG84.5CE,EURINR,0,2023-08-29,84.50,0.0025,1,CE,BCD-OPT,BCD
1,542516998,2119207,EURINR23AUG84.5PE,EURINR,0,2023-08-29,84.50,0.0025,1,PE,BCD-OPT,BCD
2,542459142,2118981,EURINR23AUG84.75CE,EURINR,0,2023-08-29,84.75,0.0025,1,CE,BCD-OPT,BCD
3,542455558,2118967,EURINR23AUG84.75PE,EURINR,0,2023-08-29,84.75,0.0025,1,PE,BCD-OPT,BCD
4,542186502,2117916,EURINR23AUG85.25CE,EURINR,0,2023-08-29,85.25,0.0025,1,CE,BCD-OPT,BCD
...,...,...,...,...,...,...,...,...,...,...,...,...
95532,2916865,11394,ZOTA,ZOTA HEALTH CARE,0,NaN,0.00,0.0500,1,EQ,NSE,NSE
95533,7436801,29050,ZUARI,ZUARI AGRO CHEMICALS,0,NaN,0.00,0.0500,1,EQ,NSE,NSE
95534,979713,3827,ZUARIIND,ZUARI INDUSTRIES,0,NaN,0.00,0.0500,1,EQ,NSE,NSE
95535,2029825,7929,ZYDUSLIFE,ZYDUS LIFESCIENCES,0,NaN,0.00,0.0500,1,EQ,NSE,NSE


In [50]:
datetime.datetime.strptime(dt_raw, "%Y%m%d")

datetime.datetime(2023, 7, 14, 0, 0)

In [ ]:
# Raw history IO
raw_eq_hist_io = 'https://www.nseindia.com/api/historical/cm/equity?symbol=SBIN&series=[%22EQ%22]&from=02-03-2023&to=21-04-2023'
raw_opt_eq_hist_io = 'https://www.nseindia.com/api/historical/fo/derivatives?&from=02-03-2023&to=21-04-2023&optionType=PE&strikePrice=500.00&expiryDate=27-Apr-2023&instrumentType=OPTSTK&symbol=SBIN'
raw_opt_index_hist_io = "https://www.nseindia.com/api/historical/fo/derivatives?symbol=NIFTY"

# Raw Index history IO
raw_ix_hist_io = 'https://niftyindices.com/Backpage.aspx/getHistoricaldatatabletoString'
data = str({'name':'NIFTY BANK','startDate':'02-Mar-2023','endDate':'21-Apr-2023'})


## Index History

In [ ]:
# Test index history. This is the only different one needing data!
ix_hist = nse_json(url=raw_ix_hist_io, data=data)
df_ix_hist = clean_index_hist(ix_hist)
df_ix_hist.head()

## Equity History

In [ ]:
# Test equity history
eq_hist = nse_json(url=raw_eq_hist_io)
df_eq_hist = clean_eq_hist(eq_hist)
df_eq_hist.head()

## Equity Option History

In [ ]:
# Test equity option history
eq_opt_json = nse_json(url=raw_opt_eq_hist_io)
df_eq_opt = clean_opt_hist(eq_opt_json)
df_eq_opt.head()

## Index Option History

In [ ]:
# Test index option history
ix_opt_json = nse_json(url=raw_opt_index_hist_io)
df_ix_opt = clean_opt_hist(ix_opt_json)
df_ix_opt.head()

# Dynamic URLs

In [1]:
import datetime
def build_url(base_url: str, params: dict) -> str:

    """Build url from base with query parameters. Preserves `[]`"""
    encoded_params = '&'.join([f"{k}={v}" for k, v in params.items()])
    url = f"{base_url}?{encoded_params}"
    
    return url

In [2]:
# Parameters
eq_symbol = 'SBIN'
idx_symbol = 'NIFTY'
days = 365
max_records = 35

# periods = days // max_records + (days % max_records > 0) + 1

end_date = datetime.datetime.now().date()


In [3]:
from support import hist_date_splits
dates = hist_date_splits(end_date)

## Index History

In [4]:
from nse import get_nse_syms
df_syms = get_nse_syms()

In [7]:
df_syms[df_syms.secType == 'IND']

,nse_symbol,ib_symbol,secType,exchange
0,NIFTY,NIFTY50,IND,NSE
1,FINNIFTY,FINNIFTY,IND,NSE
2,BANKNIFTY,BANKNIFTY,IND,NSE


In [12]:
ix_sym_map = def_dict['NSE']['INDEX_SYM_MAP']

In [14]:
{v: k for  k, v in ix_sym_map.items()}

{'NIFTY': 'NIFTY 50',
 'BANKNIFTY': 'NIFTY BANK',
 'INDIAVIX': 'INDIA VIX',
 'FINNIFTY': 'NIFTY FIN SERVICE',
 'MIDCPNIFTY': 'NIFTY MIDCAP 50'}

In [16]:
# every Thursday (FnO expiry date) over 10 years
edate = datetime.datetime.now().date()

In [22]:
sdate = edate - datetime.timedelta(365)

In [23]:
[sdate + datetime.timedelta(days=d)
    for d in range(0, (edate-sdate).days+1)
        if ((sdate + datetime.timedelta(days=d)).weekday()==3)]

[datetime.date(2022, 7, 21),
 datetime.date(2022, 7, 28),
 datetime.date(2022, 8, 4),
 datetime.date(2022, 8, 11),
 datetime.date(2022, 8, 18),
 datetime.date(2022, 8, 25),
 datetime.date(2022, 9, 1),
 datetime.date(2022, 9, 8),
 datetime.date(2022, 9, 15),
 datetime.date(2022, 9, 22),
 datetime.date(2022, 9, 29),
 datetime.date(2022, 10, 6),
 datetime.date(2022, 10, 13),
 datetime.date(2022, 10, 20),
 datetime.date(2022, 10, 27),
 datetime.date(2022, 11, 3),
 datetime.date(2022, 11, 10),
 datetime.date(2022, 11, 17),
 datetime.date(2022, 11, 24),
 datetime.date(2022, 12, 1),
 datetime.date(2022, 12, 8),
 datetime.date(2022, 12, 15),
 datetime.date(2022, 12, 22),
 datetime.date(2022, 12, 29),
 datetime.date(2023, 1, 5),
 datetime.date(2023, 1, 12),
 datetime.date(2023, 1, 19),
 datetime.date(2023, 1, 26),
 datetime.date(2023, 2, 2),
 datetime.date(2023, 2, 9),
 datetime.date(2023, 2, 16),
 datetime.date(2023, 2, 23),
 datetime.date(2023, 3, 2),
 datetime.date(2023, 3, 9),
 datetime.dat

In [ ]:
+